## 1. Configuration and Imports

In [15]:
import json
import time
import numpy as np
from numpy.linalg import norm
from pydantic import BaseModel, Field
from ollama import chat, embeddings
from pathlib import Path
from sentence_transformers import CrossEncoder

# ----- VARIABLES -----
model = "llama3:latest" #
ratio_queries = 1 # (0.1 = 10% of the dataset)
queries_type = "tool_only" #tool_param_all tool_only
reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)

# ----- FILE PATHS -----
script_folder = Path().absolute()
tools_list_path = script_folder.parent / 'input' / 'tools' / 'tools_list_exemples.json'
queries_list_path = script_folder.parent / 'input' / 'user_queries' / f'{queries_type}.json'
results_path = script_folder / 'output' / f'{queries_type}_Rerankedexempe.json'

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6301.53it/s]


## 2. RAG Engine (Retrieval-Augmented Generation)

In [16]:
def rerank_tools(user_prompt, tools_list, top_k=3):
    pairs = []
    for tool in tools_list:
        name = tool.get('name', '')
        desc = tool.get('description', '')
        tags = " ".join(tool.get('tags', []))
        examples = " ".join(tool.get('examples', [])) # <-- AJOUT CRUCIAL
        
        # On donne TOUT le contexte au Cross-Encoder
        doc_text = f"{name}: {desc}. Tags: {tags}. Examples of usage: {examples}"
        pairs.append([user_prompt, doc_text])
        
    scores = reranker_model.predict(pairs)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return [(float(scores[i]), tools_list[i]) for i in top_indices]

## 3. LLM Router Agent

In [17]:
class RouteDecision(BaseModel):
    # 1. On modifie la description pour forcer l'analyse avant le choix
    reasoning: str = Field(description="Step-by-step analysis comparing the user's request against the available tools before making a decision.")
    confidence: float = Field(description="Confidence level from 0.0 to 1.0")
    selected_tool: str = Field(description="The exact name of the tool. Return 'none' if no tool matches.")

# ----- AGENT LOGIC -----
def agent_router(user_prompt: str, relevant_tools: list, model_name: str) -> RouteDecision:
    """Passes the filtered tools and user prompt to the LLM to make the final routing decision."""
    
    # 2. Formatage des outils en texte clair (Markdown) au lieu d'un JSON brut
    tools_formatted = "\n".join([
        f"{t.get('name', 'Unknown')}: {t.get('description', '')} (Tags: {', '.join(t.get('tags', []))})" 
        for t in relevant_tools
    ])
    
    # 3. Prompt système enrichi avec des instructions strictes et des exemples (Few-Shot)
    system_prompt = f"""You are a Router Agent expert in medical and dental imaging (CBCT, IOS, MRI).
    Your role is to analyze the user's request and select the most relevant tool from the FILTERED list below.
    If none of these {len(relevant_tools)} tools fit perfectly, return 'none'.

    === FILTERED TOOLS ===
    {tools_formatted}

    === ROUTING GUIDELINES & EXAMPLES ===
    - Pay close attention to subtle differences. For example, if a user specifically asks for "batch processing" or "multiple scans", prioritize tools designed for batching (e.g., batchdentalseg).
    - If a user asks to "segment" or "split" specific teeth, ensure the tool handles instance segmentation (e.g., amasss_cli).
    - If the request is for registration, check if it's CBCT-to-CBCT, MRI-to-CBCT, or intraoral (IOS) and choose the specific tool accordingly.

    Carefully analyze the user's prompt step-by-step in the 'reasoning' field BEFORE selecting the tool. Output strictly matching the JSON schema.
    """
    
    try:
        response = chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            format=RouteDecision.model_json_schema(),
            options={"temperature": 0},
        )
        return RouteDecision.model_validate_json(response.message.content)
    except Exception as e:
        return RouteDecision(selected_tool="error", confidence=0.0, reasoning=f"Error: {str(e)}")

## 4. Data Loading and Pre-computation

In [18]:
# ----- LOAD FILES -----
with open(tools_list_path, 'r', encoding='utf-8') as f:
    tools_list = json.load(f)

with open(queries_list_path, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

## 5. Display Formatting Helpers

In [19]:
# ----- FORMATTING HELPER FUNCTIONS -----

def print_rag_block(index, total_queries, prompt, rag_icon, rag_results):
    print(f"\n{'='*65}")
    print(f"Prompt ({index}/{total_queries}) : '{prompt}'")
    print(f"{'-'*65}")
    print(f"I: CROSS-ENCODER RERANKING {rag_icon}")
    print(f"{'-'*65}")
    
    # On n'a plus qu'un seul score maintenant
    for i, (score, tool) in enumerate(rag_results, start=1):
        tool_name = tool.get('name', 'unknown_name')
        print(f"{i} Score: {score:.4f} - tool: {tool_name}")
    print(f"{'-'*65}")


def print_llm_block(decision, expected_tool, llm_status_icon, latency):
    """Handles the terminal output for the LLM routing decision."""
    print("II: LLM ROUTER DECISION")
    print(f"{'-'*65}")  
    print(f"Tool chosen: {decision.selected_tool} {llm_status_icon} ")
    print(f"Expected   : {expected_tool}")
    print(f"Confidence : {decision.confidence * 100:.2f}%")
    print(f"Latency    : {latency:.2f} seconds")
    print(f"Reasoning  : {decision.reasoning}")

## 6. Main Benchmark Loop

In [20]:
# ----- BENCHMARK INITIALIZATION -----
results_detail = []
llm_correct_count = 0
rag_correct_count = 0
total_time = 0.0

limit = max(1, int(len(queries_list) * ratio_queries))
queries_to_run = queries_list[:limit]
total_queries = len(queries_to_run)

# ----- MAIN EXECUTION LOOP -----
for index, (prompt, expected_tool) in enumerate(queries_to_run, start=1):

    t0 = time.time()
    
    # 1. Execute Reranker (On passe à top_k=3, suffisant avec cette précision)
    rag_results = rerank_tools(prompt, tools_list, top_k=3)
    
    # On extrait juste les outils (la structure de rag_results a changé)
    relevant_tools = [tool for score, tool in rag_results]    
    
    # Check RAG success & extract tool names
    rag_tool_names = [t.get('name', 'unknown') for t in relevant_tools]
    rag_hit = expected_tool in rag_tool_names
    
    if rag_hit: rag_correct_count += 1
    rag_hit_sign = "✅" if rag_hit else "❌"
    rag_prediction_str = f"{', '.join(rag_tool_names)} {rag_hit_sign}"

    print_rag_block(index, total_queries, prompt, rag_hit_sign, rag_results)

    # 2. Execute LLM Router (Llama 3 reste exactement pareil !)
    decision = agent_router(prompt, relevant_tools, model)
    t1 = time.time()
    
    latency = t1 - t0
    total_time += latency
        
    # Check LLM success & trigger print
    llm_hit = decision.selected_tool == expected_tool
    if llm_hit: llm_correct_count += 1
    llm_hit_sign = "✅" if llm_hit else "❌"

    print_llm_block(decision, expected_tool, llm_hit_sign, latency)

    # 3. Save iteration data
    results_detail.append({
        "prompt": prompt,
        "expected_tool": expected_tool,
        "rag_prediction": rag_prediction_str,
        "llm_prediction": f"{decision.selected_tool} {llm_hit_sign}",
        "Confidence": decision.confidence,
        "Latency": round(latency, 4),
        "reasoning": decision.reasoning
    })


Prompt (1/140) : 'identify landmarks on CBCT scans'
-----------------------------------------------------------------
I: CROSS-ENCODER RERANKING ✅
-----------------------------------------------------------------
1 Score: 0.9971 - tool: ali_cbct
2 Score: 0.9703 - tool: semi_aso_cbct
3 Score: 0.8726 - tool: semi_aso_ios
-----------------------------------------------------------------
II: LLM ROUTER DECISION
-----------------------------------------------------------------
Tool chosen: ali_cbct ✅ 
Expected   : ali_cbct
Confidence : 100.00%
Latency    : 2.54 seconds
Reasoning  : The user wants to identify landmarks on CBCT scans, which suggests they are looking for anatomical reference points on cone-beam CT volumes for precise dental and maxillofacial measurements and analysis. This task is best suited for the 'ali_cbct' tool, which uses AI agents to detect anatomical reference points on CBCT scans.

Prompt (2/140) : 'automatic landmark detection on intraoral scans'
-------------------

## 7. Save and Global Results

In [21]:
# ----- FINAL METRICS CALCULATION -----
total_queries = len(queries_to_run)

# Calculate both accuracies
rag_accuracy = (rag_correct_count / total_queries) * 100 if total_queries > 0 else 0.0
llm_accuracy = (llm_correct_count / total_queries) * 100 if total_queries > 0 else 0.0
avg_time = (total_time / total_queries) if total_queries > 0 else 0.0

# Structure the final output JSON report
summary = {
    "model": model,
    "metrics": {
        "total_queries": total_queries,
        "rag_correct":rag_correct_count,
        "rag_accuracy": round(rag_accuracy, 2),
        "llm_correct":llm_correct_count,
        "llm_accuracy": round(llm_accuracy, 2),
        "average_latency": round(avg_time, 4),
        "total_time": round(total_time, 4),
    },
    "details": results_detail,
}

# ----- EXPORT RESULTS -----
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

# ----- FINAL SUMMARY OUTPUT -----
print("\n BENCHMARK COMPLETED ")
print(f"RAG Accuracy (Top): {rag_accuracy:.2f}% ({rag_correct_count}/{total_queries})")
print(f"LLM Accuracy (Exact): {llm_accuracy:.2f}% ({llm_correct_count}/{total_queries})")
print(f"Average Time:         {avg_time:.2f} seconds per query")
print(f"Report saved to:      {results_path.name}")


 BENCHMARK COMPLETED 
RAG Accuracy (Top): 88.57% (124/140)
LLM Accuracy (Exact): 75.71% (106/140)
Average Time:         2.20 seconds per query
Report saved to:      tool_only_Rerankedexempe.json
